In [ ]:
import os
import matplotlib.pyplot as plt
from obspy import read, UTCDateTime
from obspy.geodetics import locations2degrees

print("Step 1: Setting event time")
event_time = UTCDateTime(2026, 5, 1, 12, 39, 55)
data_dir = r"c:\Users\kyle0\Desktop\seismoiogy_ch3\20260501123955_CWASN"

print("Step 2: Reading all Z-component SAC files")
file_pattern = os.path.join(data_dir, "*Z.D*.SAC")
st = read(file_pattern)
print("Total Z-component traces loaded:", len(st))

# 為避免重複測站，只保留每個測站最佳的 Z 分量
# 優先順序: HHZ (寬頻速度計) > EHZ (短週期速度計) > HLZ/HNZ (加速度計)
station_dict = {}
for tr in st:
    sta = tr.stats.station
    chan = tr.stats.channel
    priority = 0
    if chan.endswith("Z"):
        if chan.startswith("HH"):
            priority = 3
        elif chan.startswith("EH"):
            priority = 2
        elif chan.startswith("HL"):
            priority = 1
        else:
            priority = 0
    if sta not in station_dict or priority > station_dict[sta][1]:
        station_dict[sta] = (tr, priority)

traces = [v[0] for v in station_dict.values()]
print("Unique stations selected:", len(traces))

# 截取發震前後的波形
start_trim = event_time - 10
end_trim = event_time + 120
valid_traces_list = []
for tr in traces:
    tr.trim(start_trim, end_trim)
    if tr.stats.npts < 10 or len(tr.data) == 0:
        continue
    tr.detrend("demean")
    tr.filter("bandpass", freqmin=0.5, freqmax=5.0)
    if tr.stats.sampling_rate > 20.0:
        factor = int(tr.stats.sampling_rate / 10.0)
        if factor > 1:
            tr.decimate(factor=factor, no_filter=True)
    valid_traces_list.append(tr)
traces = valid_traces_list
print("Traces after trim:", len(traces))

# 計算距離並準備繪圖資料
plot_data = []
for tr in traces:
    dist_km = None
    if hasattr(tr.stats, "sac"):
        sac = tr.stats.sac
        if hasattr(sac, "dist") and sac.dist != -12345.0:
            dist_km = sac.dist
        elif hasattr(sac, "gcarc") and sac.gcarc != -12345.0:
            dist_km = sac.gcarc * 111.19
        elif (hasattr(sac, "stla") and hasattr(sac, "evla") and
              sac.stla != -12345.0 and sac.evla != -12345.0):
            deg = locations2degrees(sac.evla, sac.evlo, sac.stla, sac.stlo)
            dist_km = deg * 111.19

    if dist_km is not None and len(tr.data) > 0:
        max_amp = max(abs(tr.data.max()), abs(tr.data.min()))
        if max_amp > 0:
            norm_data = tr.data / max_amp
            time_offset = tr.stats.starttime - event_time
            times = tr.times() + time_offset
            plot_data.append((dist_km, norm_data, times, tr.stats.station))

plot_data.sort(key=lambda x: x[0])
print("Valid traces to plot:", len(plot_data))

# ============ 繪圖（模仿參考圖的風格） ============
print("Step 3: Plotting")

# 色彩循環（模仿參考圖的多色風格）
colors = [
    "#CC0000",   # 深紅
    "#FF8C00",   # 橘色
    "#006400",   # 深綠
    "#4169E1",   # 皇家藍
    "#8B008B",   # 深紫
    "#000000",   # 黑色
    "#B22222",   # 磚紅
    "#87CEEB",   # 淺藍
    "#228B22",   # 森林綠
    "#FF4500",   # 橘紅
]

fig, ax = plt.subplots(figsize=(14, 9))

# 標題格式模仿參考圖: 2026-05-01T12:39:55 (UTC)
ax.set_title("2026-05-01T12:39:55 (UTC)", fontsize=16, fontweight="bold", pad=30)
ax.set_xlabel("Distance (km)", fontsize=13)
ax.set_ylabel("Time (s)", fontsize=13)

amp_scale = 8.0  # 波形放大倍率

for i, (dist_km, norm_data, times, sta_name) in enumerate(plot_data):
    color = colors[i % len(colors)]
    x = dist_km + norm_data * amp_scale
    ax.plot(x, times, color=color, linewidth=0.5)
    # 在波形頂端標註測站名稱（旋轉 90 度，與參考圖一致）
    ax.text(dist_km, -2, sta_name, fontsize=5, color=color,
            ha="center", va="bottom", rotation=90, fontweight="bold")

# Y 軸設定：不反轉（0 在底部，時間向上增長，與參考圖一致）
ax.set_xlim(0, 400)
ax.set_ylim(0, 115)
ax.grid(True, linestyle="-", alpha=0.3, color="gray")

# 刻度設定
ax.tick_params(axis="both", labelsize=11)
ax.set_yticks(range(0, 120, 10))

# 白色背景
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

# 邊框
for spine in ax.spines.values():
    spine.set_edgecolor("black")
    spine.set_linewidth(0.8)

plt.show()
